# Olist Customer Retention & RFM Analysis

**Business question:** Which customer segment should an e-commerce business prioritise for retention or re-engagement, and is delivery performance associated with repeat buying?

This notebook is the reproducible walkthrough for the full code in `src/olist_analysis.py`. It intentionally uses the actual Olist files from `data/raw/` and creates outputs only when the official data is present. No example statistics are embedded here.

**Data grain / traps:** `customer_unique_id` is the person-level key; payments and reviews must be reduced to one row per order before joining; compare delivery dates rather than timestamps; late cohorts have limited follow-up.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "olist_analysis.py").exists():
    # Allow opening this notebook from notebooks/ rather than project root.
    ROOT = ROOT.parent
assert (ROOT / "src" / "olist_analysis.py").exists(), "Open this notebook from the project folder."
print("Project root:", ROOT)
print("Raw files:", sorted(p.name for p in (ROOT / "data" / "raw").glob("*.csv")))

## Run the complete validated pipeline

The pipeline loads the core CSV tables into DuckDB, profiles data, builds a one-row-per-order table, validates revenue reconciliation, computes cohorts and RFM, then tests delivery/review and delivery/repeat associations. It writes a real-data recommendation memo and aggregate visualisations.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.olist_analysis import run

results = run()
print(json.dumps({
    "customers": results["repeat_metrics"]["customers"],
    "eligible_orders": results["repeat_metrics"]["eligible_orders"],
    "repeat_customer_pct": results["repeat_metrics"]["repeat_pct"],
    "recommended_segment_candidate": results["rfm"]["recommended_segment"],
}, indent=2, default=str))

## Review generated outputs

All values below come from the data you downloaded and just analysed. Check the validation file before using any numbers in a resume or memo.

In [ ]:
processed = ROOT / "data" / "processed"
validation = json.loads((processed / "validation.json").read_text())
repeat = pd.read_csv(processed / "repeat_metrics.csv")
segments = pd.read_csv(processed / "segment_summary.csv")
delivery = pd.read_csv(processed / "delay_summary.csv")
print("Validation:", validation)
display(repeat)
display(segments)
display(delivery)

## Interpretation guardrails

- Use the selected RFM segment as a **screening candidate**, not as a validated churn prediction.
- Report effect sizes and confidence intervals. A p-value alone does not establish commercial importance.
- Delivery/review and delivery/repeat differences are observational associations, not proof of causation.
- Uplift calculations assume +1/+2/+3 percentage points of reactivation and use average historical spend; they do not include margin, costs, contactability or cannibalisation.
- Before publishing, check `docs/data_notes.md`, `docs/recommendation_memo.md`, the current Kaggle dataset licence, and the Power BI model specification.